# DC Motor Interactive Control Dashboard

This notebook adds sliders and dropdown controls for the DC motor benchmark.

You can choose the input signal shape and tune amplitude, frequency, ramp rise time, pulse duty cycle, duration, timestep, initial state and optional input-energy budget.

The dashboard can run in three modes:

```text
neural model + solver comparison
neural model only
solver only
```

The neural model is still the autonomous/no-input model. Runtime voltage is injected analytically as:

```text
x_dot = f_nn_autonomous(x) + [0, Va(t)/L]
```

This is valid for the DC motor structure because voltage enters the current derivative additively.

In [ ]:
%matplotlib inline

import csv
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output
    WIDGETS_AVAILABLE = True
except Exception as exc:
    WIDGETS_AVAILABLE = False
    print("ipywidgets is not available. Install it with: pip install ipywidgets")
    print("Original error:", repr(exc))

from datasets.dc_motor import DEFAULTS, dc_motor_gradient
from util import DynamicLoad, latest_file

## Simulation helpers

Changing the sliders does not use a pre-generated dataset. Every click on **Run simulation** creates a fresh rollout from the current UI values.

In [ ]:
DEFAULT_STABLE_MODEL = (
    "stabledynamics[latent_space_dim=2,a=0.001,projfn=NN-REHU,"
    "projfn_eps=0.01,smooth_v=0,hp=60,h=100,rehu=0.001]"
)
DEFAULT_STABLE_WEIGHT = "experiments/dc-motor-stable/checkpoint_*.pth"


def make_params():
    params = DEFAULTS.copy()
    params["Va"] = 0.0
    params["Tl"] = 0.0
    return params


def input_signal(t, kind, amplitude, frequency, rise_time, duty, input_start):
    if t < input_start:
        return 0.0
    tau = t - input_start
    if kind == "none":
        return 0.0
    if kind in ["constant", "step"]:
        return float(amplitude)
    if kind == "ramp":
        if rise_time <= 0:
            return float(amplitude)
        return float(amplitude) * min(max(tau / rise_time, 0.0), 1.0)
    if kind == "sine":
        return float(amplitude) * np.sin(2.0 * np.pi * frequency * tau)
    if kind == "pulse":
        if frequency <= 0:
            return float(amplitude)
        period = 1.0 / frequency
        phase = (tau % period) / period
        return float(amplitude) if phase < duty else 0.0
    raise ValueError(f"Unknown input kind: {kind}")


def motor_energy(x, params):
    x = np.asarray(x, dtype=np.float32)
    omega = x[..., 0]
    current = x[..., 1]
    return 0.5 * params["J"] * omega**2 + 0.5 * params["L"] * current**2


def rk4_step(rhs_fn, x, dt):
    k1 = rhs_fn(x)
    k2 = rhs_fn(x + 0.5 * dt * k1)
    k3 = rhs_fn(x + 0.5 * dt * k2)
    k4 = rhs_fn(x + dt * k3)
    return (x + (dt / 6.0) * (k1 + 2.0 * k2 + 2.0 * k3 + k4)).astype(np.float32)


def load_neural_model(model_spec=DEFAULT_STABLE_MODEL, weight_glob=DEFAULT_STABLE_WEIGHT, force_cpu=False):
    device = "cuda" if torch.cuda.is_available() and not force_cpu else "cpu"
    weight_path = latest_file(weight_glob)
    model_module = DynamicLoad("models")(model_spec)
    model = model_module.model
    model.load_state_dict(torch.load(weight_path, map_location=device))
    model.to(device)
    model.eval()
    return model, weight_path, device


def nn_autonomous_rhs(model, x_np, device):
    # stabledynamics needs gradients wrt x; do not use torch.no_grad().
    x = torch.tensor(x_np, dtype=torch.float32, device=device).reshape(1, 2)
    x = x.detach().requires_grad_(True)
    dx = model(x)
    return dx.detach().cpu().numpy().reshape(2).astype(np.float32)


def simulate_dashboard(mode, signal_kind, amplitude, frequency, rise_time, duty, input_start, duration, dt, x0_omega, x0_current, use_energy_budget, energy_budget, force_cpu=False):
    params = make_params()
    rhs_auto = dc_motor_gradient(params)

    model = None
    weight_path = None
    device = "cpu"
    if mode in ["neural", "neural_vs_solver"]:
        model, weight_path, device = load_neural_model(force_cpu=force_cpu)

    x_model = np.array([x0_omega, x0_current], dtype=np.float32)
    x_solver = x_model.copy()

    steps = int(np.ceil(duration / dt)) + 1
    t_hist, va_hist, pin_hist, ein_pos_hist, ein_net_hist = [], [], [], [], []
    x_model_hist, x_solver_hist = [], []
    cumulative_pos_energy = 0.0
    cumulative_net_energy = 0.0

    for step in range(steps):
        t = step * dt
        va = float(input_signal(t, signal_kind, amplitude, frequency, rise_time, duty, input_start))

        t_hist.append(t)
        va_hist.append(va)

        if mode in ["neural", "neural_vs_solver"]:
            current_for_power = float(x_model[1])
            x_model_hist.append(x_model.copy())
        else:
            current_for_power = float(x_solver[1])

        if mode in ["solver", "neural_vs_solver"]:
            x_solver_hist.append(x_solver.copy())

        input_power = va * current_for_power
        pin_hist.append(input_power)
        ein_pos_hist.append(cumulative_pos_energy)
        ein_net_hist.append(cumulative_net_energy)

        if use_energy_budget and cumulative_pos_energy >= energy_budget:
            break
        if step == steps - 1:
            break

        def solver_rhs(x):
            dx = rhs_auto(x).astype(np.float32)
            dx[1] += va / params["L"]
            return dx

        if mode in ["solver", "neural_vs_solver"]:
            x_solver = rk4_step(solver_rhs, x_solver, dt)

        if mode in ["neural", "neural_vs_solver"]:
            def model_rhs(x):
                dx = nn_autonomous_rhs(model, x, device)
                dx[1] += va / params["L"]
                return dx
            x_model = rk4_step(model_rhs, x_model, dt)

        cumulative_net_energy += input_power * dt
        cumulative_pos_energy += max(input_power, 0.0) * dt

    result = {
        "params": params,
        "mode": mode,
        "weight_path": weight_path,
        "device": device,
        "t": np.asarray(t_hist, dtype=np.float32),
        "va": np.asarray(va_hist, dtype=np.float32),
        "pin": np.asarray(pin_hist, dtype=np.float32),
        "ein_pos": np.asarray(ein_pos_hist, dtype=np.float32),
        "ein_net": np.asarray(ein_net_hist, dtype=np.float32),
    }

    if len(x_model_hist) > 0:
        result["x_model"] = np.asarray(x_model_hist, dtype=np.float32)
        result["energy_model"] = motor_energy(result["x_model"], params)
    if len(x_solver_hist) > 0:
        result["x_solver"] = np.asarray(x_solver_hist, dtype=np.float32)
        result["energy_solver"] = motor_energy(result["x_solver"], params)

    return result

## Plot helper

In [ ]:
def plot_dashboard_result(result):
    mode = result["mode"]
    t = result["t"]
    va = result["va"]

    fig, axes = plt.subplots(4, 2, figsize=(14, 14))
    axes = axes.ravel()

    axes[0].plot(t, va)
    axes[0].set_title("Input voltage Va(t)")
    axes[0].set_xlabel("time [s]")
    axes[0].set_ylabel("V")

    if "x_model" in result:
        xm = result["x_model"]
        axes[1].plot(t, xm[:, 0], label="model omega")
        axes[2].plot(t, xm[:, 1], label="model current")
        axes[3].semilogy(t, np.maximum(result["energy_model"], 1e-12), label="model energy")
        axes[5].plot(xm[:, 0], xm[:, 1], label="model")
        axes[5].scatter([xm[0, 0]], [xm[0, 1]], marker="o", s=40)

    if "x_solver" in result:
        xs = result["x_solver"]
        axes[1].plot(t, xs[:, 0], linestyle="--", label="solver omega")
        axes[2].plot(t, xs[:, 1], linestyle="--", label="solver current")
        axes[3].semilogy(t, np.maximum(result["energy_solver"], 1e-12), linestyle="--", label="solver energy")
        axes[5].plot(xs[:, 0], xs[:, 1], linestyle="--", label="solver")
        axes[5].scatter([xs[0, 0]], [xs[0, 1]], marker="o", s=40)

    axes[1].set_title("Angular velocity")
    axes[1].set_xlabel("time [s]")
    axes[1].set_ylabel("omega [rad/s]")

    axes[2].set_title("Current")
    axes[2].set_xlabel("time [s]")
    axes[2].set_ylabel("current [A]")

    axes[3].set_title("Physical energy")
    axes[3].set_xlabel("time [s]")
    axes[3].set_ylabel("E")

    axes[4].plot(t, result["ein_pos"], label="positive injected energy")
    axes[4].plot(t, result["ein_net"], linestyle="--", label="net injected energy")
    axes[4].set_title("Cumulative input energy")
    axes[4].set_xlabel("time [s]")
    axes[4].set_ylabel("energy")

    axes[5].scatter([0.0], [0.0], marker="x", s=100)
    axes[5].set_title("Phase portrait")
    axes[5].set_xlabel("omega [rad/s]")
    axes[5].set_ylabel("current [A]")
    axes[5].set_aspect("equal", adjustable="datalim")

    axes[6].plot(t, result["pin"])
    axes[6].set_title("Input electrical power Va(t) * current(t)")
    axes[6].set_xlabel("time [s]")
    axes[6].set_ylabel("power")

    if "x_model" in result and "x_solver" in result:
        err = result["x_solver"] - result["x_model"]
        err_norm = np.linalg.norm(err, axis=1)
        axes[7].semilogy(t, np.maximum(err_norm, 1e-12), label="||solver - model||")
        axes[7].set_title("Model vs solver state error")
        axes[7].set_xlabel("time [s]")
        axes[7].set_ylabel("error")
    else:
        axes[7].axis("off")

    for ax in axes:
        ax.grid(True, alpha=0.3)
        handles, labels = ax.get_legend_handles_labels()
        if labels:
            ax.legend(loc="best")

    fig.suptitle(f"DC motor interactive simulation | mode={mode} | weight={result.get('weight_path')} | device={result.get('device')}", y=0.995)
    fig.tight_layout()
    plt.show()

## Dashboard

Use the controls below, then press **Run simulation**.

If the neural checkpoint is missing, either train it first:

```bash
bash train_dc_motor_stable
```

or switch mode to **Solver only**.

In [ ]:
if WIDGETS_AVAILABLE:
    mode_widget = widgets.Dropdown(options=[("Neural model + solver comparison", "neural_vs_solver"), ("Neural model only", "neural"), ("Solver only", "solver")], value="neural_vs_solver", description="Mode:", style={"description_width": "initial"})
    signal_widget = widgets.Dropdown(options=["none", "constant", "step", "ramp", "sine", "pulse"], value="step", description="Signal:", style={"description_width": "initial"})
    amplitude_widget = widgets.FloatSlider(value=2.0, min=-10.0, max=10.0, step=0.1, description="Amplitude [V]:", continuous_update=False, style={"description_width": "initial"}, layout=widgets.Layout(width="520px"))
    frequency_widget = widgets.FloatSlider(value=1.0, min=0.0, max=10.0, step=0.05, description="Frequency [Hz]:", continuous_update=False, style={"description_width": "initial"}, layout=widgets.Layout(width="520px"))
    rise_time_widget = widgets.FloatSlider(value=2.0, min=0.0, max=10.0, step=0.1, description="Ramp rise time [s]:", continuous_update=False, style={"description_width": "initial"}, layout=widgets.Layout(width="520px"))
    duty_widget = widgets.FloatSlider(value=0.5, min=0.01, max=0.99, step=0.01, description="Pulse duty:", continuous_update=False, style={"description_width": "initial"}, layout=widgets.Layout(width="520px"))
    input_start_widget = widgets.FloatSlider(value=0.5, min=0.0, max=10.0, step=0.1, description="Input start [s]:", continuous_update=False, style={"description_width": "initial"}, layout=widgets.Layout(width="520px"))
    duration_widget = widgets.FloatSlider(value=20.0, min=1.0, max=60.0, step=1.0, description="Duration [s]:", continuous_update=False, style={"description_width": "initial"}, layout=widgets.Layout(width="520px"))
    dt_widget = widgets.FloatLogSlider(value=0.01, base=10, min=-4, max=-1, step=0.1, description="dt [s]:", continuous_update=False, style={"description_width": "initial"}, layout=widgets.Layout(width="520px"))
    x0_omega_widget = widgets.FloatSlider(value=0.0, min=-10.0, max=10.0, step=0.1, description="x0 omega:", continuous_update=False, style={"description_width": "initial"}, layout=widgets.Layout(width="520px"))
    x0_current_widget = widgets.FloatSlider(value=0.0, min=-10.0, max=10.0, step=0.1, description="x0 current:", continuous_update=False, style={"description_width": "initial"}, layout=widgets.Layout(width="520px"))
    use_budget_widget = widgets.Checkbox(value=False, description="Use positive input-energy budget", style={"description_width": "initial"})
    budget_widget = widgets.FloatSlider(value=1.0, min=0.0, max=20.0, step=0.1, description="Energy budget:", continuous_update=False, style={"description_width": "initial"}, layout=widgets.Layout(width="520px"))
    force_cpu_widget = widgets.Checkbox(value=False, description="Force CPU", style={"description_width": "initial"})
    run_button = widgets.Button(description="Run simulation", button_style="success", icon="play", layout=widgets.Layout(width="220px"))
    output = widgets.Output()

    controls_left = widgets.VBox([mode_widget, signal_widget, amplitude_widget, frequency_widget, rise_time_widget, duty_widget, input_start_widget])
    controls_right = widgets.VBox([duration_widget, dt_widget, x0_omega_widget, x0_current_widget, use_budget_widget, budget_widget, force_cpu_widget, run_button])
    dashboard = widgets.VBox([widgets.HBox([controls_left, controls_right]), output])

    def on_run_clicked(_):
        with output:
            clear_output(wait=True)
            try:
                result = simulate_dashboard(
                    mode=mode_widget.value,
                    signal_kind=signal_widget.value,
                    amplitude=amplitude_widget.value,
                    frequency=frequency_widget.value,
                    rise_time=rise_time_widget.value,
                    duty=duty_widget.value,
                    input_start=input_start_widget.value,
                    duration=duration_widget.value,
                    dt=dt_widget.value,
                    x0_omega=x0_omega_widget.value,
                    x0_current=x0_current_widget.value,
                    use_energy_budget=use_budget_widget.value,
                    energy_budget=budget_widget.value,
                    force_cpu=force_cpu_widget.value,
                )
                print("Simulation finished.")
                print("steps:", len(result["t"]))
                print("final t:", result["t"][-1])
                if "x_model" in result:
                    print("final model state:", result["x_model"][-1])
                    print("final model energy:", result["energy_model"][-1])
                if "x_solver" in result:
                    print("final solver state:", result["x_solver"][-1])
                    print("final solver energy:", result["energy_solver"][-1])
                print("positive injected energy:", result["ein_pos"][-1])
                print("net injected energy:", result["ein_net"][-1])
                plot_dashboard_result(result)
                globals()["LAST_INTERACTIVE_RESULT"] = result
            except Exception as exc:
                print("Simulation failed.")
                print(repr(exc))
                print()
                print("If the neural model checkpoint is missing, either:")
                print("  1) run: bash train_dc_motor_stable")
                print("  2) switch Mode to 'Solver only'")

    run_button.on_click(on_run_clicked)
    display(dashboard)
else:
    print("ipywidgets is required for this dashboard.")

## Export latest result to CSV

After running the dashboard, the latest result is stored as `LAST_INTERACTIVE_RESULT`.

In [ ]:
def save_last_result(path="experiments/dc-motor-interactive/interactive_result.csv"):
    result = globals().get("LAST_INTERACTIVE_RESULT")
    if result is None:
        print("No result yet. Run the dashboard first.")
        return

    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    has_model = "x_model" in result
    has_solver = "x_solver" in result

    header = ["t", "Va", "input_power", "positive_input_energy", "net_input_energy"]
    if has_model:
        header += ["omega_model", "current_model", "energy_model"]
    if has_solver:
        header += ["omega_solver", "current_solver", "energy_solver"]

    with path.open("w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(header)
        for i in range(len(result["t"])):
            row = [result["t"][i], result["va"][i], result["pin"][i], result["ein_pos"][i], result["ein_net"][i]]
            if has_model:
                row += [result["x_model"][i, 0], result["x_model"][i, 1], result["energy_model"][i]]
            if has_solver:
                row += [result["x_solver"][i, 0], result["x_solver"][i, 1], result["energy_solver"][i]]
            writer.writerow(row)

    print("saved:", path)

# Example after running dashboard:
# save_last_result()